# Hot Cities, Blind Sensors — Heat Where the Map Is Empty

**Standalone.** Synthetic city LST + sparse stations; metrics for heat ∩ missingness.

VI: Demo synthetic; REAL_DATA = OpenAQ + MODIS/ERA5 (xem README).

In [ ]:
%pip install -q numpy pandas matplotlib scipy

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter

RESULTS = Path('results'); RESULTS.mkdir(exist_ok=True)
RNG = np.random.default_rng(42)
GRID, N_STATIONS, RADIUS, HEAT_PCT = 64, 12, 6.0, 90

def make_city_lst(n=GRID):
    yy, xx = np.mgrid[0:n, 0:n]
    core = 32 + 8 * np.exp(-((xx - n*0.45)**2 + (yy - n*0.5)**2) / (2*12**2))
    industrial = 5 * np.exp(-((xx - n*0.75)**2 + (yy - n*0.25)**2) / (2*6**2))
    park = -4 * np.exp(-((xx - n*0.3)**2 + (yy - n*0.7)**2) / (2*8**2))
    return gaussian_filter(core + industrial + park + RNG.normal(0, 0.4, (n, n)), 1.0)

def place_stations(n, k):
    coords = []
    for _ in range(k):
        x = int(np.clip(RNG.normal(n*0.4, n*0.12), 0, n-1))
        y = int(np.clip(RNG.normal(n*0.55, n*0.12), 0, n-1))
        coords.append((y, x))
    return np.array(coords)

def coverage_mask(shape, stations, radius):
    h, w = shape; yy, xx = np.mgrid[0:h, 0:w]
    covered = np.zeros(shape, dtype=bool)
    for y, x in stations:
        covered |= (yy - y)**2 + (xx - x)**2 <= radius**2
    return covered


In [ ]:
lst = make_city_lst()
stations = place_stations(GRID, N_STATIONS)
covered = coverage_mask(lst.shape, stations, RADIUS)
thr = np.percentile(lst, HEAT_PCT)
heat, blind = lst >= thr, ~covered
blind_heat = heat & blind
frac = float(blind_heat.sum() / max(heat.sum(), 1))

# null
fracs = []
for _ in range(200):
    ys, xs = RNG.integers(0, GRID, N_STATIONS), RNG.integers(0, GRID, N_STATIONS)
    cov = coverage_mask(lst.shape, np.column_stack([ys, xs]), RADIUS)
    fracs.append(((heat & ~cov).sum()) / max(heat.sum(), 1))
null_mean = float(np.mean(fracs))

wards = np.zeros((GRID, GRID), dtype=int)
bs = GRID // 4; wid = 0
for i in range(4):
    for j in range(4):
        wards[i*bs:(i+1)*bs, j*bs:(j+1)*bs] = wid; wid += 1
rows = []
for w in range(16):
    m = wards == w
    rows.append(dict(ward=w, mean_lst=float(lst[m].mean()), frac_blind=float(blind[m].mean()),
                     frac_blind_and_heat=float((blind_heat & m).sum()/m.sum()),
                     hot_blind_score=float(lst[m].mean()*blind[m].mean())))
df = pd.DataFrame(rows).sort_values('hot_blind_score', ascending=False)
df.to_csv(RESULTS/'heat_missingness.csv', index=False)
summary = dict(mode='SYNTHETIC', frac_heat_cells_uncovered=frac, null_mean=null_mean,
               excess_vs_null=frac-null_mean)
(RESULTS/'summary.json').write_text(json.dumps(summary, indent=2))
print(summary); display(df.head())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
im0 = axes[0].imshow(lst, cmap='inferno')
axes[0].scatter(stations[:,1], stations[:,0], c='cyan', s=40, marker='^')
axes[0].set_title('LST + stations'); plt.colorbar(im0, ax=axes[0], fraction=0.046)
axes[1].imshow(heat, cmap='Reds'); axes[1].set_title('Heat extreme')
axes[2].imshow(blind_heat, cmap='Purples'); axes[2].set_title('Blind ∩ heat')
for ax in axes: ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout(); fig.savefig(RESULTS/'city_map.png', dpi=120); plt.show()
